In [1]:
import sys
sys.path.append("../")
from py_code import build_feats_csv
import numpy as np
import pandas as pd
from pathlib import Path
import glob
import os
import gc

# Build features tables from traces files for training and test

In the following snippets features are built out of extracted traces event by event. Said feature tables are meant for training and testing ML models. Different parquet builder are available in py_code/build_csv.py depending on ADC availability and characteristics. This is done four times respectively for : training on protons, testing on protons, testing on gammas, testing on single muon stations in proton showers. 

- Specify saving paths for the parquet of the features.
- Specify minimum number of photoelectrons (pe) for a station to be considered.
- Select maximum time window.
- Select features-builder function from builder_csv.py.
- **Create all required directories if they are not already present. Label each directory clearly and unambiguously, using descriptive names that reflect its purpose and contents.**
- **Make sure that pe cuts are the same for single muons and background.**
- **Make sure all other parameters are set to the same value to avoid dataset mismatches.**
- **Use same builder functions for all dataset**

In [2]:
# Specify input and save paths for training traces and training features
TRAIN_traces_input_path_smu = Path("../dfs_traces/HAWCSim_array/train/sm/100160TeV_025deg_4FF_180R175h_Black/13petrain_13petest_500ns/")
TRAIN_traces_input_path_nmu = Path("../dfs_traces/HAWCSim_array/train/bkg/100160TeV_025deg_4FF_180R175h_Black/13petrain_13petest_500ns/")

TRAIN_features_save_path_smu = "../dfs_feats/HAWCSim_array/train/sm/100160TeV_025deg_4FF_180R175h_Black/13petrain_13petest_500ns/"
TRAIN_features_save_path_nmu = "../dfs_feats/HAWCSim_array/train/bkg/100160TeV_025deg_4FF_180R175h_Black/13petrain_13petest_500ns/"

TRAIN_features_save_path_smu_bkg = "../dfs_feats/HAWCSim_array/train/sm_bkg/100160TeV_025deg_4FF_180R175h_Black/13petrain_13petest_500ns/df_mPMT_NOADC_13bkg13smu_pe.parquet"


# Specify input and save paths for test proton traces and test proton features
TEST_PROTONS_traces_input_path = Path("../dfs_traces/HAWCSim_array/test_protons/100160TeV_025deg_4FF_180R175h_Black/13petrain_13petest_500ns/")
TEST_PROTONS_features_save_path = "../dfs_feats/HAWCSim_array/test_protons/100160TeV_025deg_4FF_180R175h_Black/13petrain_13petest_500ns/"

# Specify input and save paths for test gamma traces and test gamma features
TEST_GAMMAS_traces_input_path = Path("../dfs_traces/HAWCSim_array/test_gammas/100160TeV_025deg_4FF_180R175h_Black/13petrain_13petest_500ns/")
TEST_GAMMAS_features_save_path = "../dfs_feats/_HAWCSim_array/test_gammas/100160TeV_025deg_4FF_180R175h_Black/13petrain_13petest_500ns/"

# Specify input and save paths for test proton sm-only traces and test proton sm-only features
TEST_PROTONS_SMONLY_traces_input_path = Path("../dfs_traces/HAWCSim_array/test_protons/100160TeV_025deg_4FF_180R175h_Black/13petrain_13petest_500ns/")
TEST_PROTONS_SMONLY_features_save_path = "../dfs_feats/HAWCSim_array/test_protons_SMONLY/100160TeV_025deg_4FF_180R175h_Black/13petrain_13petest_500ns/"

# Set pe threshold and time window
pe_min = 13  # minimum pe number
max_time = 500   # maximum time window

# TRAIN FEATURES (Protons)

In [3]:
for f in glob.glob(TRAIN_features_save_path_smu + "*.parquet"):        # Clean directory 
    os.remove(f)

for f in glob.glob(TRAIN_features_save_path_nmu + "*.parquet"):        # Clean directory
    os.remove(f)

for parquet_file in TRAIN_traces_input_path_smu.glob("*.parquet"):

    name = parquet_file.stem   
    print(f"Processing file: {name}") 
    parquet_file_name = str(parquet_file)
    trace_path = parquet_file_name
    
    build_feats_csv.build_df_feats_NOADC_mPMT(trace_path = trace_path, 
                                              save_path = TRAIN_features_save_path_smu + name + ".parquet",
                                              mu_frac = 0,              # Get events with pe fraction of muons above 0 (all muon events)
                                              pe_threshold = pe_min, # Set pe cut
                                              pe_lim = np.inf,          # Set pe limit
                                              ch0_present = False,      # Include central channel
                                              max_time = max_time       # Set max time window
                                              )
    gc.collect()

gc.collect()
print("All done! :))")

for parquet_file in TRAIN_traces_input_path_nmu.glob("*.parquet"):
    
    name = parquet_file.stem   
    print(f"Processing file: {name}") 
    parquet_file_name = str(parquet_file)
    trace_path = parquet_file_name
    
    build_feats_csv.build_df_feats_NOADC_mPMT(trace_path = trace_path, 
                                              save_path = TRAIN_features_save_path_nmu + name + ".parquet",
                                              mu_frac = 0,
                                              pe_threshold = pe_min,
                                              pe_lim = np.inf,
                                              ch0_present = False,
                                              max_time = max_time
                                              )
    gc.collect()
    
print("All done! :))")

gc.collect()
feats_path_smu = Path(TRAIN_features_save_path_smu)
feats_path_nmu = Path(TRAIN_features_save_path_nmu)

feats_files = list(feats_path_smu.glob("*.parquet")) + list(feats_path_nmu.glob("*.parquet"))

dfs = [pd.read_parquet(f) for f in feats_files]
combined_df = pd.concat(dfs, ignore_index = True)

combined_df.to_parquet(TRAIN_features_save_path_smu_bkg)
gc.collect()

Processing file: DAT1400013_12
Processing file: DAT1400013_9
Processing file: DAT1400076_16
Processing file: DAT1400038_20
Processing file: DAT1400038_24
Processing file: DAT1400012_1
Processing file: DAT1400037_15
Processing file: DAT1400021_15
Processing file: DAT1400092_2
Processing file: DAT1400038_1
Processing file: DAT1400092_6
Processing file: DAT1400021_2
Processing file: DAT1400021_6
Processing file: DAT1400030_11
Processing file: DAT1400043_11
Processing file: DAT1400030_15
Processing file: DAT1400087_18
Processing file: DAT1400095_2
Processing file: DAT1400081_4
Processing file: DAT1400099_3
Processing file: DAT1400020_13
Processing file: DAT1400020_17
Processing file: DAT1400030_8
Processing file: DAT1400094_11
Processing file: DAT1400031_4
Processing file: DAT1400035_10
Processing file: DAT1400076_7
Processing file: DAT1400087_11
Processing file: DAT1400075_2
Processing file: DAT1400012_11
Processing file: DAT1400012_19
Processing file: DAT1400043_10
Processing file: DAT14

0

# TEST PROTON FEATURES 

In [4]:
for f in glob.glob(TEST_PROTONS_features_save_path + "*.parquet"):
    os.remove(f)

for parquet_file in TEST_PROTONS_traces_input_path.glob("*.parquet"):
    
    name = parquet_file.stem   
    print(f"Processing file: {name}") 
    parquet_file_name = str(parquet_file)
    trace_path = parquet_file_name
    
    build_feats_csv.build_df_feats_NOADC_mPMT(trace_path = trace_path,
                                              save_path = TEST_PROTONS_features_save_path + name + ".parquet",
                                              mu_frac = 0,
                                              pe_threshold = pe_min,
                                              pe_lim = np.inf,
                                              ch0_present = False,
                                              max_time = max_time)
    gc.collect()
    
gc.collect()

Processing file: DAT1405149_18
Processing file: DAT1400239_4
Processing file: DAT1405039_13
Processing file: DAT1405091_7
Processing file: DAT1405134_7
Processing file: DAT1400218_15
Processing file: DAT1400228_25
Processing file: DAT1400216_10
Processing file: DAT1405134_19
Processing file: DAT1405085_5
Processing file: DAT1405044_12
Processing file: DAT1405013_8
Processing file: DAT1400321_12
Processing file: DAT1400200_10
Processing file: DAT1405091_3
Processing file: DAT1400230_20
Processing file: DAT1400218_19
Processing file: DAT1405065_8
Processing file: DAT1405134_11
Processing file: DAT1400239_8
Processing file: DAT1405147_11
Processing file: DAT1405003_17
Processing file: DAT1405080_17
Processing file: DAT1400234_15
Processing file: DAT1405033_23
Processing file: DAT1405147_15
Processing file: DAT1405003_13
Processing file: DAT1400234_5
Processing file: DAT1405036_5
Processing file: DAT1400247_11
Processing file: DAT1405098_12
Processing file: DAT1400220_3
Processing file: DA

0

# TEST GAMMAS FEATURES 

In [5]:
for f in glob.glob(TEST_GAMMAS_features_save_path + "*.parquet"):
    os.remove(f)

for parquet_file in TEST_GAMMAS_traces_input_path.glob("*.parquet"):
    
    name = parquet_file.stem   
    print(f"Processing file: {name}") 
    parquet_file_name = str(parquet_file)
    trace_path = parquet_file_name
    
    build_feats_csv.build_df_feats_NOADC_mPMT(trace_path = trace_path,
                                              save_path = TEST_GAMMAS_features_save_path + name + ".parquet",
                                              mu_frac = 0,
                                              pe_threshold = pe_min,
                                              pe_lim = np.inf,
                                              ch0_present = False,
                                              max_time = max_time
                                              )
    gc.collect()

gc.collect()

Processing file: DAT100488_1
Processing file: DAT100468_4
Processing file: DAT100413_5
Processing file: DAT100323_5
Processing file: DAT100015_4
Processing file: DAT100126_9
Processing file: DAT100323_1
Processing file: DAT100459_7
Processing file: DAT100488_5
Processing file: DAT100413_1
Processing file: DAT100385_10
Processing file: DAT100103_8
Processing file: DAT100491_2
Processing file: DAT100063_4
Processing file: DAT100471_7
Processing file: DAT100465_1
Processing file: DAT100046_5
Processing file: DAT100178_5
Processing file: DAT100471_3
Processing file: DAT100364_2
Processing file: DAT100406_3
Processing file: DAT100151_5
Processing file: DAT100313_2
Processing file: DAT100145_3
Processing file: DAT100293_3
Processing file: DAT100000_2
Processing file: DAT100180_3
Processing file: DAT100441_8
Processing file: DAT100014_4
Processing file: DAT100464_1
Processing file: DAT100365_6
Processing file: DAT100385_7
Processing file: DAT100441_4
Processing file: DAT100019_1
Processing fi

0

# TEST PROTONS SM-ONLY FEATURES 

In [6]:
for f in glob.glob(TEST_PROTONS_SMONLY_features_save_path + "*.parquet"):
    os.remove(f)

for parquet_file in TEST_PROTONS_SMONLY_traces_input_path.glob("*.parquet"):
    
    name = parquet_file.stem   
    print(f"Processing file: {name}") 
    parquet_file_name = str(parquet_file)
    trace_path = parquet_file_name
   
    build_feats_csv.build_df_feats_NOADC_mPMT(trace_path = trace_path,
                                            save_path = TEST_PROTONS_SMONLY_features_save_path + name + ".parquet",
                                            mu_frac = 1,
                                            pe_threshold = pe_min,
                                            pe_lim = np.inf,
                                            ch0_present = False,
                                            max_time = max_time
                                            )
    gc.collect()

gc.collect()

Processing file: DAT1405149_18
Processing file: DAT1400239_4
Processing file: DAT1405039_13
Processing file: DAT1405091_7
Processing file: DAT1405134_7
Processing file: DAT1400218_15
Processing file: DAT1400228_25
Processing file: DAT1400216_10
Processing file: DAT1405134_19
Processing file: DAT1405085_5
Processing file: DAT1405044_12
Processing file: DAT1405013_8
Processing file: DAT1400321_12
Processing file: DAT1400200_10
Processing file: DAT1405091_3
Processing file: DAT1400230_20
Processing file: DAT1400218_19
Processing file: DAT1405065_8
Processing file: DAT1405134_11
Processing file: DAT1400239_8
Processing file: DAT1405147_11
Processing file: DAT1405003_17
Processing file: DAT1405080_17
Processing file: DAT1400234_15
Processing file: DAT1405033_23
Processing file: DAT1405147_15
Processing file: DAT1405003_13
Processing file: DAT1400234_5
Processing file: DAT1405036_5
Processing file: DAT1400247_11
Processing file: DAT1405098_12
Processing file: DAT1400220_3
Processing file: DA

0